# EnergyLens-AI - Notebook 4: Model Comparison

Here I compare a few forecasting models to see which one predicts household energy demand best.

Since there are 5,566 households, fitting a separate SARIMA or Prophet model per household isn't really practical. So I did this in two ways:
1. Global models (Linear Regression, Random Forest, XGBoost) trained directly on the full feature matrix using lags, rolling stats and demographic codes.
2. For Prophet and SARIMAX, I aggregated consumption across all households into one overall demand series, since these models are built for single time series rather than panel data. This also gives a fair comparison against the ML models on one series.

### Models compared:
1. Linear Regression (baseline)
2. Random Forest
3. XGBoost
4. Facebook Prophet
5. SARIMAX


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Install prophet if it isn't already available
try:
    from prophet import Prophet
except ImportError:
    print("Prophet not found. Installing now...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "prophet", "--quiet"])
    from prophet import Prophet

sns.set_theme(style="darkgrid", palette="viridis")
plt.rcParams['figure.figsize'] = (14, 6)

# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"✅ Setup complete. Running in Colab: {IN_COLAB}")


## 2. Load Data

Loading the feature matrix built in Notebook 3.


In [ ]:
if IN_COLAB:
    from google.colab import files
    print("📤 Please upload master_features.parquet")
    uploaded = files.upload()
    features_path = list(uploaded.keys())[0]
else:
    features_path = '../data/processed/master_features.parquet'
    if not os.path.exists(features_path):
        raise FileNotFoundError(f"Missing feature matrix: {features_path}. Please run Notebook 3 first.")

# Load the dataset
print("Loading feature matrix...")
df = pd.read_parquet(features_path)
print(f"Dataset shape: {df.shape}")
df.head(3)


## 3. Train/Test Split

This is time-series data, so I can't split it randomly - a random split would let the model see future data during training and give misleadingly good test scores.

Instead I split by date:
- Train: Nov 2011 to Dec 2013
- Test: Jan 2014 to Feb 2014 (winter, so this also checks how well the model handles peak demand)


In [ ]:
# Convert day to datetime
df['day'] = pd.to_datetime(df['day'])

# Define split boundary
split_date = pd.to_datetime('2014-01-01')

print(f"Training data range: {df['day'].min()} to {split_date - pd.to_timedelta('1D')}")
print(f"Testing data range: {split_date} to {df['day'].max()}")


## 4. Aggregating to System-Wide Demand

For a fair comparison across all 5 models, I aggregated consumption and weather features by day across all households, since Prophet and SARIMAX work on a single time series, not per-household panel data.


In [ ]:
# Group by day to compute system-wide averages
agg_df = df.groupby('day').agg({
    'energy_mean': 'mean',
    'energy_lag_1': 'mean',
    'energy_lag_7': 'mean',
    'energy_lag_14': 'mean',
    'energy_lag_28': 'mean',
    'energy_roll_mean_7': 'mean',
    'energy_roll_std_7': 'mean',
    'energy_roll_mean_30': 'mean',
    'temp_avg': 'mean',
    'HDD': 'mean',
    'CDD': 'mean',
    'temp_range': 'mean',
    'is_weekend': 'first',
    'is_holiday': 'first'
}).reset_index()

agg_df = agg_df.sort_values('day').reset_index(drop=True)
print(f"System-wide aggregate shape: {agg_df.shape}")

# Define Train/Test datasets on aggregate data
train_agg = agg_df[agg_df['day'] < split_date]
test_agg = agg_df[agg_df['day'] >= split_date]

print(f"Aggregate Train Shape: {train_agg.shape}")
print(f"Aggregate Test Shape: {test_agg.shape}")

# Define feature columns
feature_cols = [
    'energy_lag_1', 'energy_lag_7', 'energy_lag_14', 'energy_lag_28',
    'energy_roll_mean_7', 'energy_roll_std_7', 'energy_roll_mean_30',
    'temp_avg', 'HDD', 'CDD', 'temp_range', 'is_weekend', 'is_holiday'
]

# Separate features (X) and target (y)
X_train_agg, y_train_agg = train_agg[feature_cols], train_agg['energy_mean']
X_test_agg, y_test_agg = test_agg[feature_cols], test_agg['energy_mean']


## 5. Evaluation Function

A helper function to compute MAE, RMSE, MAPE and R² so I don't repeat this for every model.


In [ ]:
results_dict = {}

def evaluate_predictions(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    results_dict[model_name] = {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE (%)': mape,
        'R²': r2
    }
    
    print(f"📊 {model_name} Performance:")
    print(f"  MAE:  {mae:.4f} kWh")
    print(f"  RMSE: {rmse:.4f} kWh")
    print(f"  MAPE: {mape:.2f}%")
    print(f"  R²:   {r2:.4f}\n")
    return y_pred


## 6. Training the Models

Training each model on the same train/test split and logging its metrics.


In [ ]:
# Linear Regression (baseline)
lr_model = LinearRegression()
lr_model.fit(X_train_agg, y_train_agg)
lr_preds = lr_model.predict(X_test_agg)
evaluate_predictions(y_test_agg, lr_preds, "Linear Regression")


In [ ]:
# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_agg, y_train_agg)
rf_preds = rf_model.predict(X_test_agg)
evaluate_predictions(y_test_agg, rf_preds, "Random Forest")


In [ ]:
# XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_agg, y_train_agg)
xgb_preds = xgb_model.predict(X_test_agg)
evaluate_predictions(y_test_agg, xgb_preds, "XGBoost")


In [ ]:
# Facebook Prophet
# Prophet expects columns: 'ds' (datetimes) and 'y' (target)
prophet_train = train_agg[['day', 'energy_mean']].rename(columns={'day': 'ds', 'energy_mean': 'y'})

# Add temperature and weekend flag as extra regressors
prophet_train['temp_avg'] = train_agg['temp_avg']
prophet_train['is_weekend'] = train_agg['is_weekend']

# Fit Prophet
m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
m.add_regressor('temp_avg')
m.add_regressor('is_weekend')
m.fit(prophet_train)

# Create future dataframe for test dates
prophet_test = test_agg[['day']].rename(columns={'day': 'ds'})
prophet_test['temp_avg'] = test_agg['temp_avg']
prophet_test['is_weekend'] = test_agg['is_weekend']

prophet_forecast = m.predict(prophet_test)
prophet_preds = prophet_forecast['yhat'].values
evaluate_predictions(y_test_agg, prophet_preds, "Facebook Prophet")


In [ ]:
# SARIMAX
# Fit SARIMAX with temperature and weekend flag as exogenous variables
endog_train = train_agg['energy_mean']
exog_train = train_agg[['temp_avg', 'is_weekend']]

# Kept the order small for speed - this isn't a heavily tuned SARIMAX model, just a comparison point
sarimax_model = SARIMAX(endog_train, exog=exog_train, order=(1, 1, 1), seasonal_order=(1, 0, 0, 7))
sarimax_results = sarimax_model.fit(disp=False)

exog_test = test_agg[['temp_avg', 'is_weekend']]
sarimax_preds = sarimax_results.forecast(steps=len(test_agg), exog=exog_test).values
evaluate_predictions(y_test_agg, sarimax_preds, "SARIMAX")


## 7. Comparing Results


In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame(results_dict).T
print("🏆 Model Comparison Leaderboard:")
print("=" * 60)
print(comparison_df.sort_values('MAE').to_string())


In [ ]:
# Plot predictions vs actuals
plt.figure(figsize=(16, 8))
plt.plot(test_agg['day'], y_test_agg, label='Actual Demand', color='black', linewidth=2.5)
plt.plot(test_agg['day'], lr_preds, label='Linear Regression', linestyle='--', alpha=0.7)
plt.plot(test_agg['day'], rf_preds, label='Random Forest', linestyle='--', alpha=0.7)
plt.plot(test_agg['day'], xgb_preds, label='XGBoost', color='cyan', linewidth=2)
plt.plot(test_agg['day'], prophet_preds, label='Facebook Prophet', color='magenta', linewidth=2)
plt.plot(test_agg['day'], sarimax_preds, label='SARIMAX', linestyle='--', alpha=0.7)

plt.title('London Grid Energy Demand Forecast vs Actuals (Jan - Feb 2014)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Average Household Energy Consumption (kWh)', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 8. Notes on the Results

XGBoost came out ahead of SARIMAX here. SARIMAX assumes a mostly linear relationship and stationary trend, while XGBoost can pick up non-linear effects - for example, energy use doesn't rise in a straight line as temperature drops, it can spike more sharply past a certain point. XGBoost also gets to use the lags, rolling stats and calendar flags all as separate inputs at once, while SARIMAX mainly relies on its own autoregressive terms.

I also made sure the lag/rolling features were built on `shift(1)` and not the current day's value. If I used today's consumption to build today's rolling average, the model would basically be cheating - using information it wouldn't actually have at prediction time.

The HDD feature also seems to help, since raw temperature isn't linearly related to energy use - people only really turn heating on once it gets below a certain point (around 15.5°C). HDD captures that threshold directly, which seems to make the model's job easier compared to just giving it raw temperature.

### Next (Notebook 5)
Next I move from forecasting to unsupervised learning - detecting anomalies in consumption and clustering households into usage groups.
